## **BAB 1: PENDAHULUAN DAN PERSIAPAN LINGKUNGAN (Introduction and Environment Setup)**

### **1.1. Deskripsi Arsitektur Sistem (System Architecture Description)**

Sistem *Retrieval-Augmented Generation* (RAG) tingkat lanjut ini dirancang untuk mengatasi keterbatasan memori statis pada *Large Language Models* (LLM). Arsitektur ini mengintegrasikan model penalaran hukum yang telah dilatih sebelumnya (GRPO) dengan basis pengetahuan eksternal yang dinamis berupa dokumen peraturan perundang-undangan Indonesia.

Alur kerja sistem ini mengadopsi paradigma *Advanced RAG* yang terdiri dari:
1.  **Ingestion Pipeline:** Pemrosesan dokumen PDF menjadi potongan teks hierarkis (*Parent-Child Chunking*) yang diperkaya dengan metadata struktural.
2.  **Retrieval Pipeline:** Penggunaan *Ensemble Retriever* yang menggabungkan pencarian leksikal (BM25) dan pencarian semantik (ChromaDB), diperkuat dengan teknik *Hypothetical Document Embeddings* (HyDE) untuk mengatasi ambiguitas *query*.
3.  **Reranking & Fallback:** Penilaian ulang dokumen menggunakan *Cross-Encoder* untuk memastikan relevansi absolut, dengan mekanisme *fallback* ke mesin pencari internet (DuckDuckGo) jika dokumen lokal tidak memadai.
4.  **Generation Pipeline:** Sintesis jawaban akhir oleh model GRPO yang diinstruksikan untuk melakukan penalaran logis (`<think>`) berdasarkan konteks yang ditarik, disertai dengan penyematan sitasi sumber.


### **1.2. Instalasi Dependensi Kritis (Critical Dependency Installation)**

Tahap ini mengorkestrasi instalasi ekosistem pustaka yang dibutuhkan untuk membangun *pipeline* RAG. Penggunaan `subprocess` memastikan instalasi berjalan secara terisolasi dan log keluaran ditekan untuk menjaga kebersihan lingkungan kerja.

In [1]:
import sys
import subprocess
import time

def install_rag_dependencies():
    """
    Melakukan instalasi pustaka dependensi utama untuk sistem Advanced RAG.
    Menggabungkan seluruh kebutuhan (LangChain, Chroma, Reranker, Gradio, DDGS) 
    ke dalam satu eksekusi batch untuk efisiensi waktu dan mencegah konflik versi.
    """
    print("[SYSTEM] Memulai instalasi dependensi ekosistem RAG...")
    
    # Daftar komprehensif seluruh pustaka yang dibutuhkan
    dependencies = [
        "unsloth",                  # Kernel optimasi inferensi LLM
        "langchain",                # Orkestrasi pipeline RAG
        "langchain-community",      # Integrasi pihak ketiga
        "langchain-core",           # Komponen inti LangChain
        "langchain-classic",        # Arsitektur retriever tingkat lanjut (Parent-Child, Ensemble)
        "langchain-chroma",         # Integrasi ChromaDB modern
        "langchain-text-splitters", # Utilitas pemotongan teks
        "langchain-huggingface",    # Integrasi model Hugging Face
        "pypdf",                    # Ekstraksi teks dari dokumen PDF
        "chromadb",                 # Basis data vektor lokal
        "rank_bm25",                # Algoritma pencarian leksikal
        "sentence-transformers",    # Model embedding dan reranking
        "duckduckgo-search",        # Mekanisme fallback pencarian internet
        "ddgs",                     # Wrapper spesifik untuk DuckDuckGo
        "python-dotenv",            # Manajemen variabel lingkungan
        "gradio"                    # Antarmuka pengguna interaktif (UI)
    ]
    
    # Eksekusi instalasi dengan menekan log peringatan pip
    command = [sys.executable, "-m", "pip", "install", "-U"] + dependencies + ["--quiet", "--disable-pip-version-check"]
    
    try:
        process_result = subprocess.run(command, capture_output=True, text=True)
        if process_result.returncode == 0:
            print("[SUCCESS] Seluruh dependensi RAG dan UI berhasil diinstal.")
        else:
            print(f"[FATAL ERROR] Kegagalan instalasi: {process_result.stderr}")
            raise RuntimeError("Instalasi dependensi gagal.")
    except Exception as execution_error:
        print(f"[FATAL ERROR] Interupsi sistem saat instalasi: {str(execution_error)}")
        raise execution_error

# Eksekusi instalasi
installation_start = time.time()
install_rag_dependencies()
installation_end = time.time()
print(f"[SYSTEM] Operasi instalasi diselesaikan dalam {installation_end - installation_start:.2f} detik.")
print("[SYSTEM] Lanjutkan ke sel berikutnya untuk memuat library ke dalam memori.")

[SYSTEM] Memulai instalasi dependensi ekosistem RAG...
[SUCCESS] Seluruh dependensi RAG dan UI berhasil diinstal.
[SYSTEM] Operasi instalasi diselesaikan dalam 60.02 detik.
[SYSTEM] Lanjutkan ke sel berikutnya untuk memuat library ke dalam memori.


In [2]:
# 1. STANDARD LIBRARY & TYPING
import os
import sys
import gc
import re
import time
import pickle
import warnings
import logging
from typing import List, Tuple, Dict, Any, Union, Optional

# 2. CORE MACHINE LEARNING & UTILITIES
import torch
import chromadb
import gradio as gr
from transformers import logging as hf_logging
from kaggle_secrets import UserSecretsClient

# Menekan peringatan non-kritis untuk menjaga kebersihan log terminal
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
hf_logging.set_verbosity_error()

# 3. LANGCHAIN CORE & COMMUNITY (Data Processing & Basic Retrieval)
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# 4. LANGCHAIN CLASSIC & CHROMA (Advanced Retrieval Architecture)
from langchain_chroma import Chroma
from langchain_classic.storage import InMemoryStore
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

# 5. UNSLOTH (LLM Generator Optimization)
from unsloth import FastLanguageModel, get_chat_template

print("[SUCCESS] Seluruh modul dan pustaka berhasil dimuat ke dalam memori aktif.")

/tmp/ipykernel_59/1129396131.py:29: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[SUCCESS] Seluruh modul dan pustaka berhasil dimuat ke dalam memori aktif.


### **1.3. Inisialisasi Lingkungan dan Pemuatan Model Generator (Environment and Generator Initialization)**

Tahap ini memuat pustaka ke dalam memori aktif dan menginisialisasi model GRPO yang telah dilatih pada fase sebelumnya. Model ini akan bertindak sebagai *Generator* utama dalam arsitektur RAG.


In [3]:
# Menekan peringatan non-kritis dari pustaka pihak ketiga
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def setup_credentials():
    """
    Mengambil token autentikasi dari Kaggle Secrets dan menyimpannya 
    sebagai variabel lingkungan untuk digunakan oleh pustaka Hugging Face.
    """
    try:
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
        os.environ["HF_TOKEN"] = hf_token
        print("[SUCCESS] Kredensial Hugging Face berhasil dimuat.")
    except Exception as e:
        print(f"[WARNING] Gagal memuat Kaggle Secrets: {str(e)}")
        print("[INFO] Pastikan HF_TOKEN telah dikonfigurasi di menu Add-ons.")

def initialize_generator_model(repo_id: str, max_seq_len: int = 2048):
    """
    Memuat model GRPO (Generator) dan tokenizer ke dalam memori GPU.
    Menggunakan optimasi Unsloth untuk inferensi yang efisien.
    """
    print(f"[SYSTEM] Memuat model generator dari repositori: {repo_id}")
    
    from unsloth import FastLanguageModel
    from langchain_huggingface import HuggingFacePipeline
    from transformers import pipeline
    
    # Pembersihan VRAM sebelum alokasi model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Memuat model dengan kuantisasi 4-bit untuk efisiensi memori
    base_model, active_tokenizer = FastLanguageModel.from_pretrained(
        model_name=repo_id,
        max_seq_length=max_seq_len,
        load_in_4bit=True,
        fast_inference=False, # Dinonaktifkan untuk kompatibilitas Kaggle
        device_map="auto"
    )
    
    # Mengaktifkan mode inferensi native Unsloth (2x lebih cepat)
    FastLanguageModel.for_inference(base_model)
    
    print("[SYSTEM] Mengonfigurasi pipeline generasi teks...")
    
    # PERBAIKAN: Menghapus tokenizer_kwargs yang menyebabkan ValueError
    text_generation_pipeline = pipeline(
        task="text-generation",
        model=base_model,
        tokenizer=active_tokenizer,
        max_new_tokens=400, # Dibatasi agar model tidak looping
        temperature=0.05,    # Suhu rendah untuk determinisme
        top_p=0.9,
        repetition_penalty=1.15, # Penalti repetisi untuk mencegah pengulangan kata
        return_full_text=False,
        pad_token_id=active_tokenizer.eos_token_id
    )
    
    # Membungkus pipeline ke dalam antarmuka LangChain
    langchain_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
    
    print("[SUCCESS] Model generator berhasil diinisialisasi dan diintegrasikan dengan LangChain.")
    return langchain_llm, active_tokenizer

# Eksekusi inisialisasi
setup_credentials()

# # Menggunakan model hasil GRPO dari fase sebelumnya
# generator_repo_id = "latief18/legal-llama-3.1-grpo-reasoning-merged"

# Menggunakan model hasil GRPO dari fase sebelumnya
generator_repo_id = "latief18/legal-llama-3.1-sft-optimized"

llm_generator, tokenizer = initialize_generator_model(repo_id=generator_repo_id)

[SUCCESS] Kredensial Hugging Face berhasil dimuat.
[SYSTEM] Memuat model generator dari repositori: latief18/legal-llama-3.1-sft-optimized
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/975 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

[SYSTEM] Mengonfigurasi pipeline generasi teks...
[SUCCESS] Model generator berhasil diinisialisasi dan diintegrasikan dengan LangChain.


## **BAB 2: PEMROSESAN DOKUMEN DAN PENGAYAAN METADATA (Document Processing and Metadata Enrichment)**

### **2.1. Pemuatan Dokumen Hukum (Document Ingestion)**

Tahap ini bertanggung jawab untuk membaca file PDF mentah dari direktori lokal dan mengekstrak teksnya ke dalam format objek dokumen yang dapat diproses oleh LangChain. Penggunaan `PyPDFLoader` dipilih karena kemampuannya mempertahankan struktur halaman, yang sangat krusial untuk pelacakan sitasi nantinya.

In [4]:
def find_pdf_directory(base_path: str = "/kaggle/input/") -> str:
    """
    Melakukan pencarian rekursif untuk menemukan direktori yang secara fisik 
    mengandung file berekstensi .pdf. Mengatasi masalah sanitasi nama direktori 
    pada lingkungan Kaggle.

    Args:
        base_path (str): Direktori akar untuk pencarian.

    Returns:
        str: Path absolut menuju direktori target.
    """
    print(f"[SYSTEM] Memulai pemindaian rekursif pada: {base_path}")
    
    for root, dirs, files in os.walk(base_path):
        if any(file_name.lower().endswith(".pdf") for file_name in files):
            print(f"[SUCCESS] Direktori dataset tervalidasi pada: {root}")
            return root
            
    raise FileNotFoundError("[FATAL ERROR] Tidak ditemukan file PDF pada seluruh sub-direktori input.")

def ingest_legal_documents(directory_path: str) -> List[Document]:
    """
    Mengekstrak teks dari seluruh file PDF di dalam direktori target.
    Dilengkapi dengan mekanisme audit karakter untuk mendeteksi kegagalan 
    ekstraksi pada halaman yang mengalami korupsi format atau hasil scan gambar.

    Args:
        directory_path (str): Path absolut direktori dataset.

    Returns:
        List[Document]: Daftar objek dokumen mentah.
    """
    print(f"[SYSTEM] Menginisialisasi ekstraksi teks dari: {directory_path}")
    
    pdf_files = [os.path.join(directory_path, f) for f in os.listdir(directory_path) if f.lower().endswith('.pdf')]
    
    if not pdf_files:
        raise ValueError("[FATAL ERROR] Direktori valid, namun daftar file PDF kosong.")

    raw_documents = []
    total_characters = 0
    
    for file_path in pdf_files:
        file_name = os.path.basename(file_path)
        try:
            loader = PyPDFLoader(file_path)
            pages = loader.load()
            
            # Audit kualitas ekstraksi
            file_char_count = sum(len(page.page_content) for page in pages)
            total_characters += file_char_count
            
            if file_char_count < 100:
                print(f"[WARNING] Ekstraksi mencurigakan pada {file_name}. Hanya {file_char_count} karakter terbaca.")
            else:
                print(f"[LOG] Ekstraksi sukses: {file_name} | {len(pages)} Halaman | {file_char_count:,} Karakter")
                
            raw_documents.extend(pages)
            
        except Exception as extraction_error:
            print(f"[ERROR] Kegagalan pemrosesan pada {file_name}: {str(extraction_error)}")

    print(f"[SUCCESS] Ingestion selesai. Total {len(raw_documents)} halaman ({total_characters:,} karakter) dimuat ke memori.")
    return raw_documents

# Eksekusi Pemuatan Dokumen
dataset_directory = find_pdf_directory()
raw_docs = ingest_legal_documents(dataset_directory)

[SYSTEM] Memulai pemindaian rekursif pada: /kaggle/input/
[SUCCESS] Direktori dataset tervalidasi pada: /kaggle/input/datasets/latief18/dokumen-uud-rag-pgabl
[SYSTEM] Menginisialisasi ekstraksi teks dari: /kaggle/input/datasets/latief18/dokumen-uud-rag-pgabl
[LOG] Ekstraksi sukses: PP Nomor 5 Tahun 2021.pdf | 739 Halaman | 520,047 Karakter
[LOG] Ekstraksi sukses: UU Nomor 6 Tahun 2023.pdf | 1127 Halaman | 1,331,462 Karakter
[LOG] Ekstraksi sukses: PP Nomor 51 Tahun 2023.pdf | 27 Halaman | 39,686 Karakter
[LOG] Ekstraksi sukses: PP Nomor 35 Tahun 2021.pdf | 56 Halaman | 68,966 Karakter
[SUCCESS] Ingestion selesai. Total 1949 halaman (1,960,161 karakter) dimuat ke memori.


### **2.2. Pengayaan Metadata Dokumen (Metadata Enrichment)**

Dokumen hukum memiliki hierarki dan identitas yang kaku. Fungsi ini menggunakan *Regular Expressions* (Regex) untuk mengekstrak informasi spesifik dari nama file dan menyuntikkannya ke dalam metadata setiap halaman. Metadata ini adalah kunci untuk melakukan *filtering* yang akurat pada tahap pencarian (Bab 4).

In [5]:
def sanitize_and_enrich_metadata(documents: List[Document]) -> List[Document]:
    """
    Membersihkan metadata bawaan dari informasi yang tidak relevan (junk metadata) 
    dan menyuntikkan atribut terstruktur berdasarkan nomenklatur file hukum Indonesia.
    Integritas tipe data dijaga secara ketat untuk mencegah kegagalan filtering di basis data vektor.

    Args:
        documents (List[Document]): Daftar dokumen mentah hasil ekstraksi.

    Returns:
        List[Document]: Daftar dokumen dengan metadata yang telah disanitasi dan diperkaya.
    """
    print("[SYSTEM] Memulai protokol sanitasi dan pengayaan metadata...")
    
    enriched_documents = []
    junk_keys = ['producer', 'creator', 'author', 'creationdate', 'moddate']
    
    for doc in documents:
        # 1. Sanitasi Metadata (Menghapus polusi data)
        for key in junk_keys:
            doc.metadata.pop(key, None)
            
        source_path = doc.metadata.get("source", "")
        file_name = os.path.basename(source_path)
        
        # 2. Ekstraksi Tahun (Validasi rentang tahun logis 1900-2099)
        year_match = re.search(r'\b(19|20)\d{2}\b', file_name)
        regulation_year = int(year_match.group(0)) if year_match else None
        
        # 3. Ekstraksi Jenis Peraturan
        file_name_upper = file_name.upper()
        if file_name_upper.startswith("UU"):
            regulation_type = "Undang-Undang"
        elif file_name_upper.startswith("PP"):
            regulation_type = "Peraturan Pemerintah"
        elif file_name_upper.startswith("PERPPU"):
            regulation_type = "Peraturan Pemerintah Pengganti Undang-Undang"
        else:
            regulation_type = "Peraturan Lainnya"
            
        # 4. Ekstraksi Nomor Peraturan
        number_match = re.search(r'(?i)nomor\s+(\d+)', file_name)
        regulation_number = int(number_match.group(1)) if number_match else None
        
        # 5. Konstruksi Sitasi Standar
        page_num = doc.metadata.get("page", 0) + 1
        citation_format = f"{file_name.replace('.pdf', '')}, Halaman {page_num}"
        
        # Injeksi atribut terstruktur
        doc.metadata.update({
            "file_name": file_name,
            "regulation_type": regulation_type,
            "regulation_number": regulation_number,
            "regulation_year": regulation_year,
            "citation": citation_format
        })
        
        enriched_documents.append(doc)
        
    print("[SUCCESS] Metadata berhasil disanitasi dan diperkaya.")
    
    # Verifikasi integritas skema metadata pada sampel pertama
    if enriched_documents:
        print("\n[LOG] Audit Skema Metadata (Sampel Dokumen 1):")
        for key, value in enriched_documents[0].metadata.items():
            print(f"  -> {key} ({type(value).__name__}): {value}")
            
    return enriched_documents

# Eksekusi Pengayaan Metadata
enriched_docs = sanitize_and_enrich_metadata(raw_docs)

[SYSTEM] Memulai protokol sanitasi dan pengayaan metadata...
[SUCCESS] Metadata berhasil disanitasi dan diperkaya.

[LOG] Audit Skema Metadata (Sampel Dokumen 1):
  -> source (str): /kaggle/input/datasets/latief18/dokumen-uud-rag-pgabl/PP Nomor 5 Tahun 2021.pdf
  -> total_pages (int): 739
  -> page (int): 0
  -> page_label (str): 1
  -> file_name (str): PP Nomor 5 Tahun 2021.pdf
  -> regulation_type (str): Peraturan Pemerintah
  -> regulation_number (int): 5
  -> regulation_year (int): 2021
  -> citation (str): PP Nomor 5 Tahun 2021, Halaman 1


### **2.3. Strategi Pemotongan Dokumen Hierarkis (Parent-Child Chunking Strategy)**

Pemotongan teks (*chunking*) standar sering kali merusak konteks kalimat hukum yang panjang. Strategi *Parent-Child* memecahkan masalah ini dengan membuat potongan kecil (*Child*) yang optimal untuk pencarian vektor matematis, namun tetap mempertahankan tautan ke potongan besar (*Parent*) yang akan diserahkan ke LLM agar konteks hukumnya tidak terputus.

In [6]:
def execute_hierarchical_chunking(documents: List[Document]) -> Tuple[List[Document], RecursiveCharacterTextSplitter]:
    """
    Menerapkan arsitektur pemotongan teks dua tingkat (Parent-Child).
    Menggunakan separator spesifik hukum Indonesia untuk mencegah pemotongan 
    di tengah-tengah definisi pasal atau ayat yang krusial.

    Args:
        documents (List[Document]): Daftar dokumen dengan metadata terstruktur.

    Returns:
        Tuple[List[Document], RecursiveCharacterTextSplitter]: 
        Kumpulan Parent Chunks dan objek pemotong untuk Child Chunks.
    """
    print("[SYSTEM] Menginisialisasi arsitektur pemotongan hierarkis...")
    
    # Definisi separator spesifik untuk dokumen hukum Indonesia
    # Memprioritaskan pemotongan pada pergantian BAB, Bagian, atau Pasal
    legal_separators = [
        "\nBAB ", 
        "\nBagian ", 
        "\nPasal ", 
        "\n\n", 
        "\n", 
        ". ", 
        " "
    ]
    
    # 1. Konfigurasi Parent Splitter
    # Kapasitas ditingkatkan ke 3000 untuk mengakomodasi pasal yang panjang
    parent_chunk_size = 3000
    parent_chunk_overlap = 300
    
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=parent_chunk_size,
        chunk_overlap=parent_chunk_overlap,
        separators=legal_separators,
        keep_separator=True
    )
    
    # 2. Konfigurasi Child Splitter
    # Kapasitas kecil untuk presisi representasi vektor matematis
    child_chunk_size = 400
    child_chunk_overlap = 50
    
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=child_chunk_size,
        chunk_overlap=child_chunk_overlap,
        separators=legal_separators,
        keep_separator=True
    )
    
    print(f"[LOG] Konfigurasi Parent : Size={parent_chunk_size}, Overlap={parent_chunk_overlap}")
    print(f"[LOG] Konfigurasi Child  : Size={child_chunk_size}, Overlap={child_chunk_overlap}")
    
    # Eksekusi pemotongan tingkat makro (Parent)
    parent_chunks = parent_splitter.split_documents(documents)
    
    print(f"[SUCCESS] Pemotongan makro selesai. Menghasilkan {len(parent_chunks)} Parent Chunks.")
    
    # Audit rasio pemotongan untuk mendeteksi anomali
    average_chunk_per_page = len(parent_chunks) / len(documents) if documents else 0
    print(f"[LOG] Rasio pemotongan: {average_chunk_per_page:.2f} chunks per halaman.")
    
    return parent_chunks, child_splitter

# Eksekusi Pemotongan Hierarkis
parent_documents, child_text_splitter = execute_hierarchical_chunking(enriched_docs)

[SYSTEM] Menginisialisasi arsitektur pemotongan hierarkis...
[LOG] Konfigurasi Parent : Size=3000, Overlap=300
[LOG] Konfigurasi Child  : Size=400, Overlap=50
[SUCCESS] Pemotongan makro selesai. Menghasilkan 1949 Parent Chunks.
[LOG] Rasio pemotongan: 1.00 chunks per halaman.


## **BAB 3: PEMBANGUNAN BASIS DATA VEKTOR (Vector Database Construction)**

### **3.1. Inisialisasi Model Embedding Sumber Terbuka (Open-Source Embedding Initialization)**

Tahap ini memuat model *embedding* yang bertugas mengubah teks menjadi representasi vektor matematis berdimensi tinggi. Pemilihan model `BAAI/bge-m3` sangat strategis karena model ini mendukung multibahasa (termasuk Bahasa Indonesia) dan memiliki arsitektur *dense retrieval* yang sangat kuat untuk dokumen formal/hukum.

In [7]:
def initialize_embedding_model(model_name: str = "BAAI/bge-m3") -> HuggingFaceEmbeddings:
    """
    Memuat model embedding open-source ke dalam memori GPU untuk akselerasi komputasi.
    Model ini akan digunakan untuk mengubah teks dokumen dan query pengguna menjadi vektor.

    Args:
        model_name (str): Repositori model embedding di Hugging Face.

    Returns:
        HuggingFaceEmbeddings: Objek embedding yang siap digunakan oleh LangChain.
    """
    print(f"[SYSTEM] Menginisialisasi model embedding: {model_name}")
    
    # Mendeteksi ketersediaan GPU untuk akselerasi
    compute_device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[LOG] Perangkat komputasi yang digunakan: {compute_device.upper()}")
    
    try:
        # Konfigurasi model embedding
        embedding_engine = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': compute_device},
            encode_kwargs={'normalize_embeddings': True} # Normalisasi L2 untuk akurasi Cosine Similarity
        )
        print("[SUCCESS] Model embedding berhasil dimuat ke dalam memori.")
        return embedding_engine
        
    except Exception as initialization_error:
        print(f"[FATAL ERROR] Kegagalan inisialisasi model embedding: {str(initialization_error)}")
        raise initialization_error

# Eksekusi inisialisasi embedding
embedding_model = initialize_embedding_model()

[SYSTEM] Menginisialisasi model embedding: BAAI/bge-m3
[LOG] Perangkat komputasi yang digunakan: CUDA


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

[SUCCESS] Model embedding berhasil dimuat ke dalam memori.


### **3.2. Penyimpanan Vektor Lokal dan Pemetaan Relasi Induk-Anak (Local Vector Storage and Parent-Child Mapping)**

Sub-bab ini adalah inti dari arsitektur *Advanced RAG*. Kita tidak menyimpan potongan besar (*Parent Chunks*) langsung ke dalam basis data vektor karena akan menurunkan akurasi pencarian. Sebaliknya, kita memotong *Parent Chunks* menjadi *Child Chunks* menggunakan `child_text_splitter` dari Bab 2. 

*Child Chunks* diubah menjadi vektor dan disimpan di ChromaDB, sementara *Parent Chunks* aslinya disimpan di memori lokal (`InMemoryByteStore`). Saat pencarian terjadi, sistem akan menemukan *Child* yang relevan, lalu mengembalikan *Parent*-nya secara utuh kepada LLM.


In [8]:
def manage_doc_store_persistence(doc_store, file_path: str, mode: str = "save"):
    """
    Mengelola persistensi data InMemoryStore menggunakan protokol Pickle.
    Mencegah kehilangan teks asli (Parent Documents) saat sesi kernel berakhir.
    """
    if mode == "save":
        print(f"[SYSTEM] Menyimpan Parent Store ke: {file_path}")
        with open(file_path, 'wb') as f:
            # Mengambil dictionary internal dari InMemoryStore
            pickle.dump(doc_store.store, f)
        print("[SUCCESS] Parent Store berhasil diamankan di disk.")
        
    elif mode == "load":
        if os.path.exists(file_path):
            print(f"[SYSTEM] Memuat Parent Store dari: {file_path}")
            with open(file_path, 'rb') as f:
                loaded_data = pickle.load(f)
                doc_store.store = loaded_data
            print(f"[SUCCESS] {len(loaded_data)} Parent Documents berhasil dipulihkan.")
            return True
        return False

def construct_vector_database(parent_docs: List[Document], child_splitter, embedding_engine, persist_dir: str):
    """
    Membangun basis data vektor hierarkis dengan kontrol koneksi tingkat rendah.
    Menggunakan PersistentClient untuk menghindari SQLite locking issues di Kaggle.
    """
    print("[SYSTEM] Memulai protokol konstruksi database vektor...")
    
    # 1. Force Cleanup: Menghapus residu dan membebaskan memori
    if os.path.exists(persist_dir):
        print(f"[LOG] Menghapus indeks lama di: {persist_dir}")
        shutil.rmtree(persist_dir, ignore_errors=True)
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        # 2. Inisialisasi Native Chroma Client
        # Ini memastikan kita memiliki kontrol penuh atas file database
        native_db_client = chromadb.PersistentClient(path=persist_dir)
        
        # 3. Inisialisasi Wrapper LangChain Chroma
        vector_store = Chroma(
            client=native_db_client,
            collection_name="legal_knowledge_index",
            embedding_function=embedding_engine
        )
        
        # 4. Inisialisasi Document Store (Parent)
        doc_store = InMemoryStore()

        # Cek apakah data lama bisa dipulihkan untuk menghemat waktu
        if manage_doc_store_persistence(doc_store, "/kaggle/working/parent_store.pkl", mode="load"):
            print("[INFO] Menggunakan data Parent Store yang sudah ada. Melewati proses embedding ulang.")
            return hierarchical_retriever
        
        # 5. Konstruksi Retriever
        hierarchical_retriever = ParentDocumentRetriever(
            vectorstore=vector_store,
            docstore=doc_store,
            child_splitter=child_splitter,
            search_kwargs={"k": 5}
        )

        # 6. Eksekusi Batch Indexing dengan Proteksi Koneksi
        batch_size = 50 # Ukuran batch lebih kecil untuk stabilitas maksimal
        total_docs = len(parent_docs)
        
        print(f"[LOG] Memulai embedding massal ({total_docs} dokumen)...")
        
        for i in range(0, total_docs, batch_size):
            batch = parent_docs[i : i + batch_size]
            hierarchical_retriever.add_documents(batch, ids=None)
            
            # Verifikasi progres
            current_count = vector_store._collection.count()
            print(f"  -> Progress: {min(i + batch_size, total_docs)}/{total_docs} dokumen | Vektor: {current_count:,}")

        parent_store_path = "/kaggle/working/parent_store.pkl"
        manage_doc_store_persistence(doc_store, parent_store_path, mode="save")

        print("[SUCCESS] Basis data vektor berhasil dikunci dan diindeks.")
        return hierarchical_retriever

    except Exception as e:
        print(f"[FATAL ERROR] Gagal membangun database: {str(e)}")
        # Jika gagal, pastikan folder dihapus agar tidak mengunci sesi berikutnya
        if os.path.exists(persist_dir):
            shutil.rmtree(persist_dir, ignore_errors=True)
        raise e

# Konfigurasi path
chroma_persist_directory = "/kaggle/working/chroma_legal_db"

# Eksekusi
parent_child_retriever = construct_vector_database(
    parent_docs=parent_documents,
    child_splitter=child_text_splitter,
    embedding_engine=embedding_model,
    persist_dir=chroma_persist_directory
)

[SYSTEM] Memulai protokol konstruksi database vektor...
[LOG] Memulai embedding massal (1949 dokumen)...
  -> Progress: 50/1949 dokumen | Vektor: 251
  -> Progress: 100/1949 dokumen | Vektor: 520
  -> Progress: 150/1949 dokumen | Vektor: 784
  -> Progress: 200/1949 dokumen | Vektor: 1,047
  -> Progress: 250/1949 dokumen | Vektor: 1,308
  -> Progress: 300/1949 dokumen | Vektor: 1,575
  -> Progress: 350/1949 dokumen | Vektor: 1,832
  -> Progress: 400/1949 dokumen | Vektor: 1,882
  -> Progress: 450/1949 dokumen | Vektor: 1,932
  -> Progress: 500/1949 dokumen | Vektor: 1,982
  -> Progress: 550/1949 dokumen | Vektor: 2,032
  -> Progress: 600/1949 dokumen | Vektor: 2,082
  -> Progress: 650/1949 dokumen | Vektor: 2,132
  -> Progress: 700/1949 dokumen | Vektor: 2,182
  -> Progress: 750/1949 dokumen | Vektor: 2,287
  -> Progress: 800/1949 dokumen | Vektor: 2,551
  -> Progress: 850/1949 dokumen | Vektor: 2,792
  -> Progress: 900/1949 dokumen | Vektor: 3,038
  -> Progress: 950/1949 dokumen | Vekt

## **BAB 4: REKAYASA PENCARIAN TINGKAT LANJUT (Advanced Retrieval Engineering)**

### **4.1. Pencarian Leksikal (Lexical Retrieval)**

Tahap ini menginisialisasi mesin pencari berbasis kata kunci menggunakan algoritma BM25 (Best Matching 25). Algoritma ini sangat krusial dalam domain hukum karena mampu menangkap kecocokan eksak pada terminologi spesifik (seperti "Pasal 15", "PKWT", atau "Pesangon") yang sering kali terlewatkan oleh pencarian semantik murni.


In [9]:
def initialize_lexical_retriever(documents: List[Document], top_k: int = 5) -> BM25Retriever:
    """
    Membangun indeks pencarian leksikal menggunakan algoritma BM25.
    Menggunakan Parent Documents agar konteks kata kunci yang ditemukan 
    tetap berada dalam satu kesatuan paragraf/pasal yang utuh.

    Args:
        documents (List[Document]): Daftar dokumen induk (Parent Chunks).
        top_k (int): Jumlah dokumen maksimal yang dikembalikan.

    Returns:
        BM25Retriever: Objek retriever berbasis frekuensi kata.
    """
    print("[SYSTEM] Menginisialisasi mesin pencari leksikal (BM25)...")
    
    if not documents:
        raise ValueError("[FATAL ERROR] Daftar dokumen kosong. Tidak dapat membangun indeks BM25.")

    try:
        # Membangun indeks BM25 dari sekumpulan dokumen
        bm25_engine = BM25Retriever.from_documents(documents)
        
        # Mengatur batas jumlah dokumen yang dikembalikan
        bm25_engine.k = top_k
        
        print(f"[SUCCESS] Indeks BM25 berhasil dibangun dari {len(documents)} dokumen.")
        return bm25_engine
        
    except Exception as indexing_error:
        print(f"[FATAL ERROR] Kegagalan saat membangun indeks BM25: {str(indexing_error)}")
        raise indexing_error

# Eksekusi inisialisasi BM25
# Menggunakan parent_documents dari Bab 2 agar selaras dengan Semantic Retriever
lexical_retriever = initialize_lexical_retriever(parent_documents, top_k=5)

[SYSTEM] Menginisialisasi mesin pencari leksikal (BM25)...
[SUCCESS] Indeks BM25 berhasil dibangun dari 1949 dokumen.


### **4.2. Pencarian Semantik (Semantic Retrieval)**

Pencarian semantik telah diinisialisasi secara implisit pada Bab 3 melalui `ParentDocumentRetriever`. Pada sub-bab ini, kita hanya melakukan ekstraksi dan konfigurasi ulang parameter pencarian agar selaras dengan arsitektur *Ensemble*.

In [10]:
def configure_semantic_retriever(hierarchical_retriever, top_k: int = 5):
    """
    Mengonfigurasi ulang parameter pencarian pada ParentDocumentRetriever 
    yang telah dibangun di Bab 3.

    Args:
        hierarchical_retriever: Objek ParentDocumentRetriever dari Bab 3.
        top_k (int): Jumlah dokumen maksimal yang dikembalikan.

    Returns:
        Retriever: Objek retriever semantik yang telah dikonfigurasi.
    """
    print("[SYSTEM] Mengonfigurasi parameter mesin pencari semantik...")
    
    # Memodifikasi argumen pencarian internal
    hierarchical_retriever.search_kwargs = {"k": top_k}
    
    print(f"[SUCCESS] Semantic Retriever dikonfigurasi untuk mengembalikan Top-{top_k} dokumen.")
    return hierarchical_retriever

# Eksekusi konfigurasi
# Menggunakan parent_child_retriever dari Bab 3
semantic_retriever = configure_semantic_retriever(parent_child_retriever, top_k=5)

[SYSTEM] Mengonfigurasi parameter mesin pencari semantik...
[SUCCESS] Semantic Retriever dikonfigurasi untuk mengembalikan Top-5 dokumen.


### **4.3. Penggabungan Pencarian (Ensemble Retriever)**

Fusi antara BM25 dan Semantic Retriever dengan penentuan bobot spesifik (misal: 0.4 untuk BM25, 0.6 untuk Semantic).
Pengaturan parameter untuk mengambil setidaknyaFusi antara pencarian leksikal dan semantik dilakukan menggunakan `EnsembleRetriever`. Pembobotan (weights) diatur secara asimetris: 40% untuk BM25 (menangkap kata kunci eksak) dan 60% untuk Semantik (menangkap makna kontekstual). Kombinasi ini terbukti secara empiris memberikan *Recall* tertinggi pada dokumen legal.
 5 dokumen teratas (Retrieve Top-5).

In [11]:
# PERBAIKAN KRITIS: Menyelaraskan import dengan arsitektur langchain-classic v0.3+
try:
    from langchain_classic.retrievers import EnsembleRetriever
    print("[SUCCESS] Modul EnsembleRetriever berhasil dimuat melalui langchain_classic.")
except ImportError:
    try:
        # Fallback ke path standar jika lingkungan berbeda
        from langchain.retrievers import EnsembleRetriever
        print("[SUCCESS] Modul EnsembleRetriever dimuat melalui path standar.")
    except ImportError:
        print("[FATAL ERROR] Modul 'EnsembleRetriever' tidak ditemukan di sistem.")
        print("[HELP] Pastikan Bab 1.2 (Instalasi langchain-classic) telah dijalankan dan session di-restart.")
        raise

def build_ensemble_retriever(lexical_engine, semantic_engine, weights: List[float] = [0.4, 0.6]) -> EnsembleRetriever:
    """
    Menggabungkan paradigma pencarian leksikal (BM25) dan semantik (Vector) 
    menggunakan algoritma Reciprocal Rank Fusion (RRF). 
    Metode ini menjamin keseimbangan antara akurasi kata kunci dan pemahaman konteks.

    Args:
        lexical_engine: Objek BM25Retriever yang sudah terindeks.
        semantic_engine: Objek ParentDocumentRetriever yang sudah terindeks.
        weights (List[float]): Distribusi bobot [Leksikal, Semantik].

    Returns:
        EnsembleRetriever: Engine pencarian hibrida yang siap digunakan.
    """
    print("[SYSTEM] Membangun arsitektur Ensemble Retriever (Hybrid Search)...")
    
    # Validasi matematis bobot
    if abs(sum(weights) - 1.0) > 1e-6:
        raise ValueError("[FATAL ERROR] Akumulasi bobot Ensemble harus bernilai tepat 1.0")

    try:
        # Inisialisasi Hybrid Search Engine
        hybrid_retriever = EnsembleRetriever(
            retrievers=[lexical_engine, semantic_engine],
            weights=weights
        )
        
        print(f"[SUCCESS] Hybrid Search Engine aktif.")
        print(f"[LOG] Konfigurasi Bobot: BM25 ({weights[0]}) | Semantic ({weights[1]})")
        return hybrid_retriever
        
    except Exception as ensemble_error:
        print(f"[FATAL ERROR] Kegagalan fusi retriever: {str(ensemble_error)}")
        raise ensemble_error

# Eksekusi pembangunan Ensemble Retriever
# Menggunakan 'lexical_retriever' dari Bab 4.1 dan 'semantic_retriever' dari Bab 4.2
hybrid_search_engine = build_ensemble_retriever(
    lexical_engine=lexical_retriever, 
    semantic_engine=semantic_retriever,
    weights=[0.4, 0.6] # Prioritas lebih tinggi pada pencarian semantik
)

[SUCCESS] Modul EnsembleRetriever berhasil dimuat melalui langchain_classic.
[SYSTEM] Membangun arsitektur Ensemble Retriever (Hybrid Search)...
[SUCCESS] Hybrid Search Engine aktif.
[LOG] Konfigurasi Bobot: BM25 (0.4) | Semantic (0.6)


### **4.4. Penyaringan Berbasis Metadata (Metadata Filtering)**

Fungsi ini bertindak sebagai *pre-processing layer* sebelum *query* dikirim ke mesin pencari. Fungsi ini menggunakan *Regular Expressions* untuk mendeteksi penyebutan tahun atau nomor peraturan di dalam pertanyaan pengguna, lalu membangun objek filter yang akan mempersempit ruang pencarian ChromaDB secara drastis.


In [12]:
def extract_metadata_filters(user_query: str) -> Dict[str, Any]:
    """
    Menganalisis pertanyaan pengguna untuk mengekstrak entitas hukum (Tahun/Nomor)
    dan mengonversinya menjadi format filter metadata yang dipahami oleh ChromaDB.

    Args:
        user_query (str): Pertanyaan mentah dari pengguna.

    Returns:
        Dict[str, Any]: Objek filter metadata. Mengembalikan dict kosong jika tidak ada entitas.
    """
    filters = {}
    
    # 1. Deteksi Tahun (Contoh: "tahun 2021")
    year_match = re.search(r'(?i)tahun\s+(20\d{2})', user_query)
    if year_match:
        extracted_year = int(year_match.group(1))
        filters["regulation_year"] = extracted_year
        print(f"[FILTER DETECTED] Tahun: {extracted_year}")

    # 2. Deteksi Nomor Peraturan (Contoh: "PP 35" atau "Nomor 35")
    number_match = re.search(r'(?i)(?:pp|uu|nomor|no\.?)\s*(\d+)', user_query)
    if number_match:
        extracted_number = int(number_match.group(1))
        filters["regulation_number"] = extracted_number
        print(f"[FILTER DETECTED] Nomor Peraturan: {extracted_number}")

    # Konstruksi sintaks filter ChromaDB ($and operator jika ada >1 filter)
    if len(filters) > 1:
        chroma_filter = {"$and": [{k: v} for k, v in filters.items()]}
    elif len(filters) == 1:
        chroma_filter = filters
    else:
        chroma_filter = {}

    return chroma_filter

def execute_filtered_search(query: str, ensemble_engine, semantic_engine) -> List[Document]:
    """
    Mengeksekusi pencarian dengan menerapkan filter metadata secara dinamis.
    Catatan Arsitektural: BM25 tidak mendukung pre-filtering metadata secara native 
    di LangChain, sehingga filter hanya diaplikasikan pada Semantic Engine.

    Args:
        query (str): Pertanyaan pengguna.
        ensemble_engine: Mesin pencari hibrida (tanpa filter).
        semantic_engine: Mesin pencari semantik (mendukung filter).

    Returns:
        List[Document]: Daftar dokumen hasil pencarian.
    """
    print(f"\n[SYSTEM] Memproses Query: '{query}'")
    
    # Ekstraksi filter dari query
    dynamic_filter = extract_metadata_filters(query)
    
    if dynamic_filter:
        print(f"[LOG] Menerapkan Metadata Filter: {dynamic_filter}")
        # Injeksi filter ke dalam kwargs Semantic Retriever
        semantic_engine.search_kwargs["filter"] = dynamic_filter
        
        # Eksekusi pencarian menggunakan Ensemble (Semantic akan terfilter, BM25 tetap global)
        retrieved_docs = ensemble_engine.invoke(query)
        
        # Reset filter setelah pencarian agar tidak mengkontaminasi query berikutnya
        semantic_engine.search_kwargs.pop("filter", None)
    else:
        print("[LOG] Tidak ada filter terdeteksi. Melakukan pencarian global.")
        retrieved_docs = ensemble_engine.invoke(query)
        
    print(f"[SUCCESS] Berhasil mengambil {len(retrieved_docs)} dokumen relevan.")
    return retrieved_docs

# --- UJI COBA SISTEM PENCARIAN ---
print("\n--- AUDIT SISTEM PENCARIAN TINGKAT LANJUT ---")

# Test Case 1: Pencarian Global (Tanpa Filter)
test_query_1 = "Apa definisi dari Perjanjian Kerja Waktu Tertentu?"
results_1 = execute_filtered_search(test_query_1, hybrid_search_engine, semantic_retriever)

# Test Case 2: Pencarian Spesifik (Dengan Filter Tahun dan Nomor)
test_query_2 = "Berapa pesangon untuk PHK karena efisiensi menurut PP 35 Tahun 2021?"
results_2 = execute_filtered_search(test_query_2, hybrid_search_engine, semantic_retriever)


--- AUDIT SISTEM PENCARIAN TINGKAT LANJUT ---

[SYSTEM] Memproses Query: 'Apa definisi dari Perjanjian Kerja Waktu Tertentu?'
[LOG] Tidak ada filter terdeteksi. Melakukan pencarian global.
[SUCCESS] Berhasil mengambil 7 dokumen relevan.

[SYSTEM] Memproses Query: 'Berapa pesangon untuk PHK karena efisiensi menurut PP 35 Tahun 2021?'
[FILTER DETECTED] Tahun: 2021
[FILTER DETECTED] Nomor Peraturan: 35
[LOG] Menerapkan Metadata Filter: {'$and': [{'regulation_year': 2021}, {'regulation_number': 35}]}
[SUCCESS] Berhasil mengambil 7 dokumen relevan.


## **BAB 5: TRANSFORMASI QUERY DAN PENILAIAN ULANG (Query Transformation and Reranking)**

### **5.1. Pembuatan Dokumen Hipotetis (Hypothetical Document Embeddings - HyDE)**

Tahap ini mengimplementasikan teknik HyDE untuk mengatasi masalah *vocabulary mismatch* antara pertanyaan pengguna yang singkat dengan dokumen hukum yang kaku. Kita menggunakan LLM GRPO yang telah dimuat di Bab 1 untuk menghasilkan jawaban hipotetis (halusinasi terarah) sebelum melakukan pencarian vektor.

In [13]:
def build_hyde_chain(llm_engine):
    """
    Membangun pipeline LangChain untuk menghasilkan dokumen hipotetis (HyDE).
    Sesuai kriteria Advanced, LLM diinstruksikan untuk menghasilkan 2 variasi jawaban.

    Args:
        llm_engine: Objek HuggingFacePipeline (LLM GRPO) dari Bab 1.

    Returns:
        Runnable: Chain LangChain yang siap dieksekusi.
    """
    print("[SYSTEM] Menginisialisasi arsitektur Hypothetical Document Embeddings (HyDE)...")
    
    # Prompt dirancang khusus untuk memicu penalaran hukum sebelum menjawab
    hyde_prompt_template = """
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    Anda adalah asisten hukum. Tugas Anda adalah menghasilkan 2 (dua) paragraf jawaban hipotetis 
    yang sangat formal dan menggunakan terminologi hukum Indonesia untuk menjawab pertanyaan pengguna. 
    Jangan berikan pengantar, langsung tuliskan 2 variasi jawaban tersebut.<|eot_id|>
    <|start_header_id|>user<|end_header_id|>
    Pertanyaan: {question}<|eot_id|>
    <|start_header_id|>assistant<|end_header_id|>
    """
    
    prompt = PromptTemplate.from_template(hyde_prompt_template)
    
    # Membangun chain: Input -> Prompt -> LLM -> String Output
    hyde_chain = (
        {"question": RunnablePassthrough()} 
        | prompt 
        | llm_engine 
        | StrOutputParser()
    )
    
    print("[SUCCESS] Pipeline HyDE berhasil dikonfigurasi.")
    return hyde_chain

def generate_hypothetical_documents(query: str, hyde_chain) -> List[str]:
    """
    Mengeksekusi chain HyDE untuk menghasilkan teks hipotetis.
    """
    print(f"[LOG] Menghasilkan dokumen hipotetis untuk query: '{query}'")
    try:
        hypothetical_text = hyde_chain.invoke(query)
        # Menggabungkan query asli dengan teks hipotetis untuk memperkaya vektor pencarian
        augmented_query = f"{query}\n\n{hypothetical_text}"
        return augmented_query
    except Exception as e:
        print(f"[WARNING] Kegagalan generasi HyDE: {str(e)}. Fallback ke query asli.")
        return query

# Eksekusi inisialisasi HyDE
# Membutuhkan 'llm_generator' dari Bab 1.3
hyde_generator_chain = build_hyde_chain(llm_generator)

[SYSTEM] Menginisialisasi arsitektur Hypothetical Document Embeddings (HyDE)...
[SUCCESS] Pipeline HyDE berhasil dikonfigurasi.


### **5.2. Penilaian Ulang Dokumen (Cross-Encoder Reranking)**

Pencarian awal (Bab 4) mungkin mengembalikan dokumen yang memiliki kata kunci sama tetapi konteksnya salah. Tahap ini menggunakan model *Cross-Encoder* untuk menilai ulang (skoring) pasangan `(Query, Dokumen)` secara komprehensif dan mengurutkannya kembali.

In [14]:
# PERBAIKAN KRITIS: Menyelaraskan import dengan arsitektur langchain-classic v0.3+
try:
    from langchain_classic.retrievers import ContextualCompressionRetriever
    from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
    from langchain_community.cross_encoders import HuggingFaceCrossEncoder
    print("[SUCCESS] Modul Reranker berhasil dimuat melalui langchain_classic.")
except ImportError:
    try:
        # Fallback ke path standar jika lingkungan berbeda
        from langchain.retrievers import ContextualCompressionRetriever
        from langchain.retrievers.document_compressors import CrossEncoderReranker
        from langchain_community.cross_encoders import HuggingFaceCrossEncoder
        print("[SUCCESS] Modul Reranker dimuat melalui path standar.")
    except ImportError:
        print("[FATAL ERROR] Modul Reranking tidak ditemukan di sistem.")
        raise

def initialize_reranker_pipeline(base_retriever, top_k: int = 3):
    """
    Membangun pipeline Reranking menggunakan model Cross-Encoder.
    Model ini akan menilai ulang hasil dari Ensemble Retriever dan memotongnya 
    menjadi Top-K dokumen paling relevan berdasarkan skor semantik yang ketat.

    Args:
        base_retriever: Objek EnsembleRetriever dari Bab 4.
        top_k (int): Jumlah dokumen final yang akan diteruskan ke LLM.

    Returns:
        ContextualCompressionRetriever: Retriever yang telah dilengkapi dengan Reranker.
    """
    print("[SYSTEM] Menginisialisasi arsitektur Cross-Encoder Reranking...")
    
    try:
        # Menggunakan model reranker BGE yang sangat akurat untuk Bahasa Indonesia
        reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
        
        # Mengonfigurasi kompresor untuk mengambil Top-K dokumen
        compressor = CrossEncoderReranker(model=reranker_model, top_n=top_k)
        
        # Membungkus base retriever dengan kompresor reranker
        compression_retriever = ContextualCompressionRetriever(
            base_compressor=compressor,
            base_retriever=base_retriever
        )
        
        print(f"[SUCCESS] Reranker aktif. Dikonfigurasi untuk mengekstrak Top-{top_k} dokumen.")
        return compression_retriever
        
    except Exception as reranker_error:
        print(f"[FATAL ERROR] Kegagalan inisialisasi Reranker: {str(reranker_error)}")
        raise reranker_error

# Eksekusi inisialisasi Reranker
# Menggunakan 'hybrid_search_engine' yang telah sukses dibangun di Bab 4.3
advanced_reranker_engine = initialize_reranker_pipeline(hybrid_search_engine, top_k=3)

[SUCCESS] Modul Reranker berhasil dimuat melalui langchain_classic.
[SYSTEM] Menginisialisasi arsitektur Cross-Encoder Reranking...


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

[SUCCESS] Reranker aktif. Dikonfigurasi untuk mengekstrak Top-3 dokumen.


### **5.3. Ekstraksi Skor Relevansi dan Mekanisme Cadangan (Relevance Score Extraction and Fallback Mechanism)**

Tahap ini adalah lapisan pertahanan terakhir. Jika skor relevansi dari dokumen terbaik (Top-1) hasil *Reranker* berada di bawah ambang batas (*threshold*), sistem akan menyimpulkan bahwa basis data lokal tidak memiliki jawaban, lalu secara otomatis mengalihkan pencarian ke internet menggunakan DuckDuckGo.

In [15]:
# SUPRESI PERINGATAN: Membungkam peringatan max_new_tokens dari transformers
hf_logging.set_verbosity_error()

from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_core.documents import Document
from typing import Dict, Any

def execute_advanced_retrieval_with_fallback(query: str, hyde_chain, reranker_engine, threshold: float = 0.0) -> Dict[str, Any]:
    """
    Mengorkestrasi pipeline RAG dengan perbaikan metode skoring Cross-Encoder.
    Menggunakan .score() untuk mendapatkan nilai logit relevansi secara akurat.
    """
    print(f"\n[SYSTEM] Memproses Query: '{query}'")
    
    # 1. Generasi Dokumen Hipotetis (HyDE)
    augmented_query = hyde_chain.invoke(query)
    
    # 2. Retrieval Tahap Awal (Hybrid Search)
    print("[LOG] Mengambil kandidat dokumen dari database lokal...")
    base_docs = reranker_engine.base_retriever.invoke(augmented_query)
    
    if not base_docs:
        print("[WARNING] Database lokal kosong. Memicu Fallback...")
        return trigger_internet_fallback(query)

    # 3. Manual Reranking dengan Metode .score()
    print(f"[LOG] Menilai ulang {len(base_docs)} kandidat dokumen...")
    
    # Menyiapkan pasangan [Query, Teks Dokumen]
    pairs = [[query, doc.page_content] for doc in base_docs]
    
    try:
        # PERBAIKAN: Menggunakan .score() sesuai dokumentasi HuggingFaceCrossEncoder
        scores = reranker_engine.base_compressor.model.score(pairs)
        
        # Menyuntikkan skor ke metadata untuk keperluan sortir dan audit
        for doc, score in zip(base_docs, scores):
            doc.metadata["relevance_score"] = float(score)
        
        # Sortir dokumen berdasarkan skor tertinggi
        sorted_docs = sorted(base_docs, key=lambda x: x.metadata["relevance_score"], reverse=True)
        
        top_k_docs = sorted_docs[:3]
        top_score = top_k_docs[0].metadata["relevance_score"]
        
        print(f"[LOG] Skor Relevansi Tertinggi (Logit): {top_score:.4f} (Threshold: {threshold})")

        # 4. Logika Kondisional Fallback
        if top_score < threshold:
            print(f"[SYSTEM] Skor {top_score:.4f} di bawah threshold. Beralih ke internet...")
            return trigger_internet_fallback(query)
        
        print(f"[SUCCESS] Dokumen lokal tervalidasi (Skor: {top_score:.4f}).")
        return {
            "source": "Lokal (Database Hukum)",
            "documents": top_k_docs,
            "score": top_score
        }
        
    except Exception as e:
        print(f"[ERROR] Kegagalan pada tahap skoring: {str(e)}")
        return trigger_internet_fallback(query)

def trigger_internet_fallback(query: str) -> Dict[str, Any]:
    """
    Mengeksekusi pencarian internet dengan penanganan error import dinamis.
    """
    print("[SYSTEM] Menjalankan prosedur pencarian internet (DuckDuckGo)...")
    try:
        # Menggunakan pemanggilan langsung untuk menghindari masalah namespace ddgs
        from langchain_community.tools import DuckDuckGoSearchRun
        from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
        
        search_wrapper = DuckDuckGoSearchAPIWrapper(max_results=3)
        ddg_search = DuckDuckGoSearchRun(api_wrapper=search_wrapper)
        
        search_result = ddg_search.run(query)
        
        internet_doc = Document(
            page_content=search_result,
            metadata={
                "source": "Internet (DuckDuckGo)",
                "citation": "Referensi Pencarian Internet"
            }
        )
        
        return {
            "source": "Internet (DuckDuckGo)",
            "documents": [internet_doc],
            "score": 999.0
        }
    except Exception as e:
        print(f"[FATAL ERROR] Mekanisme fallback gagal total: {str(e)}")
        return {"source": "Error", "documents": [], "score": 0.0}
        
# --- EKSEKUSI AUDIT FINAL (5 TEST CASES) ---
print("\n" + "="*60)
print("PENGUJIAN PIPELINE ADVANCED RAG (5 TEST CASES)")
print("="*60)

test_cases = [
    "Berapa pesangon untuk PHK karena efisiensi menurut PP 35 Tahun 2021?", # Eksplisit Hukum
    "Apa syarat sahnya sebuah perjanjian kerja waktu tertentu?",             # Implisit Hukum
    "Siapa Presiden Indonesia yang menjabat pada tahun 2024?",              # Fakta Umum
    "Berapa harga emas antam 1 gram hari ini?",                             # Berita Terkini
    "Bagaimana resep dan cara membuat kue bolu panggang yang lembut?"       # Ambigu/Luar Domain
]

results = []
for i, test_query in enumerate(test_cases, 1):
    print(f"\n--- TEST CASE {i} ---")
    res = execute_advanced_retrieval_with_fallback(
        query=test_query, 
        hyde_chain=hyde_generator_chain, 
        reranker_engine=advanced_reranker_engine,
        threshold=0.4 # Threshold logit: > 0 relevan, < 0 tidak relevan
    )
    results.append(res)


PENGUJIAN PIPELINE ADVANCED RAG (5 TEST CASES)

--- TEST CASE 1 ---

[SYSTEM] Memproses Query: 'Berapa pesangon untuk PHK karena efisiensi menurut PP 35 Tahun 2021?'
[LOG] Mengambil kandidat dokumen dari database lokal...
[LOG] Menilai ulang 8 kandidat dokumen...
[LOG] Skor Relevansi Tertinggi (Logit): 0.4150 (Threshold: 0.4)
[SUCCESS] Dokumen lokal tervalidasi (Skor: 0.4150).

--- TEST CASE 2 ---

[SYSTEM] Memproses Query: 'Apa syarat sahnya sebuah perjanjian kerja waktu tertentu?'
[LOG] Mengambil kandidat dokumen dari database lokal...
[LOG] Menilai ulang 6 kandidat dokumen...
[LOG] Skor Relevansi Tertinggi (Logit): 0.9953 (Threshold: 0.4)
[SUCCESS] Dokumen lokal tervalidasi (Skor: 0.9953).

--- TEST CASE 3 ---

[SYSTEM] Memproses Query: 'Siapa Presiden Indonesia yang menjabat pada tahun 2024?'
[LOG] Mengambil kandidat dokumen dari database lokal...
[LOG] Menilai ulang 8 kandidat dokumen...
[LOG] Skor Relevansi Tertinggi (Logit): 0.0851 (Threshold: 0.4)
[SYSTEM] Skor 0.0851 di bawah

## **BAB 6: GENERASI JAWABAN DAN SITASI (Answer Generation and Citation)**

### **6.1. Rekayasa Prompt Kontekstual (Contextual Prompt Engineering)**

Tahap ini merancang instruksi sistem yang mengikat model GRPO agar beroperasi secara ketat di dalam batasan konteks yang diberikan (RAG). Prompt ini secara eksplisit memaksa model untuk mempertahankan kemampuan penalarannya (`<think>`) yang telah dilatih pada fase *Reinforcement Learning*, sekaligus mencegah halusinasi informasi.

In [16]:
def build_rag_generation_chain(llm_engine):
    """
    Membangun pipeline generasi dengan Ultra-Short Prompt.
    Instruksi yang sangat pendek dan tegas terbukti lebih efektif untuk model 8B 
    dalam mencegah halusinasi dan memastikan kepatuhan pada konteks RAG.
    """
    print("[SYSTEM] Mengonfigurasi ulang arsitektur Contextual Prompt Engineering (Ultra-Short Mode)...")
    
    # Prompt dipangkas habis. Tidak ada contoh, tidak ada aturan panjang.
    # Hanya instruksi absolut untuk menggunakan konteks dan format yang diminta.
    rag_prompt_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Anda adalah asisten hukum. Jawab HANYA pakai KONTEKS di bawah. 
Format: <think>[analisis]</think> Jawaban: [hasil].<|eot_id|>
<|start_header_id|>user<|end_header_id|>
KONTEKS:
{context}

PERTANYAAN: {question}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
<think>""" 

    prompt_compiler = PromptTemplate.from_template(rag_prompt_template)
    
    # Stop words agresif untuk memotong generasi jika model mencoba keluar dari jalur
    llm_with_stop = llm_engine.bind(stop=["<|eot_id|>", "KONTEKS:", "PERTANYAAN:", "User:", "Assistant:"])
    
    generation_chain = (
        prompt_compiler 
        | llm_with_stop 
        | StrOutputParser()
    )
    
    print("[SUCCESS] RAG Generation Chain (Ultra-Short Reasoning) berhasil dikonfigurasi.")
    return generation_chain

# Inisialisasi ulang chain generasi
rag_chain = build_rag_generation_chain(llm_generator)

[SYSTEM] Mengonfigurasi ulang arsitektur Contextual Prompt Engineering (Ultra-Short Mode)...
[SUCCESS] RAG Generation Chain (Ultra-Short Reasoning) berhasil dikonfigurasi.


### **6.2. Eksekusi Inferensi LLM, Ekstraksi dan Penyematan Sitasi**

Sub-bab ini digabungkan ke dalam satu fungsi orkestrasi utama. Fungsi ini menerima *output* dari *retriever* (Bab 5), memformat dokumen menjadi satu kesatuan teks konteks, mengeksekusi inferensi LLM, dan secara otomatis mengekstrak metadata sitasi untuk disematkan di akhir jawaban sebagai bentuk pertanggungjawaban data (*data provenance*).

In [17]:
import re
from typing import List, Dict, Any
from langchain_core.documents import Document

def format_retrieved_context(documents: List[Document]) -> str:
    """Menggabungkan konten dokumen dengan batas karakter ketat."""
    formatted_context = []
    for index, doc in enumerate(documents, 1):
        # Batas 1000 karakter per dokumen agar model tidak kewalahan membaca
        content = doc.page_content.strip()[:1000] 
        formatted_context.append(f"--- Dokumen {index} ---\n{content}\n")
    return "\n".join(formatted_context)

def extract_and_format_citations(documents: List[Document], source_type: str) -> str:
    """Mengekstrak dan memformat sitasi maksimal 3 sumber teratas."""
    if source_type == "Internet (DuckDuckGo)":
        return "\n\n**Referensi Sumber:**\n1. Informasi Publik Internet (DuckDuckGo Search)\n"

    citations = []
    for doc in documents:
        citation = doc.metadata.get("citation", "Sumber Tidak Diketahui")
        if citation not in citations:
            citations.append(citation)
            
    if not citations:
        return ""
        
    formatted_citations = "\n\n**Referensi Sumber:**\n"
    for idx, citation in enumerate(citations[:3], 1): 
        formatted_citations += f"{idx}. {citation}\n"
        
    return formatted_citations

def clean_model_output(raw_output: str) -> str:
    """
    Sanitasi ekstrem untuk memastikan format <think> dan Jawaban Akhir sempurna.
    Memotong kalimat yang menggantung dan menghapus repetisi instruksi.
    """
    # 1. Injeksi tag pembuka (karena kita memancingnya di prompt)
    cleaned_text = f"<think>{raw_output.strip()}"
    
    # 2. Hapus karakter non-ASCII (sampah generasi)
    cleaned_text = re.sub(r'[^\x00-\x7F]+', ' ', cleaned_text)
    
    # 3. Potong jika model mulai mengulang-ulang tag sistem atau kata assistant
    cleaned_text = cleaned_text.split("<|")[0].split("assistant")[0].split("user")[0]
    
    # 4. Penanganan tag </think> yang hilang
    if "</think>" not in cleaned_text:
        if "Jawaban:" in cleaned_text:
            cleaned_text = cleaned_text.replace("Jawaban:", "</think>\n\nJawaban:", 1)
        else:
            cleaned_text += "\n</think>\n\nJawaban: Maaf, proses generasi terpotong sebelum memberikan jawaban akhir."
            return cleaned_text.strip()

    # 5. Pemotongan Agresif (Mencegah Looping dan Halusinasi Lanjutan)
    if "Jawaban:" in cleaned_text:
        parts = cleaned_text.split("Jawaban:")
        # Ambil bagian sebelum Jawaban (think) dan isi Jawaban pertama saja
        clean_answer = parts[1].strip()
        
        # Potong kalimat yang menggantung di akhir (Truncation Fix)
        last_period_idx = clean_answer.rfind('.')
        if last_period_idx != -1:
            clean_answer = clean_answer[:last_period_idx + 1]
            
        cleaned_text = f"{parts[0]}Jawaban: {clean_answer}"

    return cleaned_text.strip()

def execute_rag_generation_pipeline(query: str, retrieval_result: Dict[str, Any], generation_chain) -> str:
    """
    Mengorkestrasi generasi jawaban, pembersihan halusinasi, dan penyematan sitasi.
    """
    print(f"\n[SYSTEM] Memulai fase generasi untuk: '{query[:50]}...'")
    
    if not isinstance(retrieval_result, dict):
        return "[FATAL ERROR] Format data retrieval tidak valid."
        
    retrieved_docs = retrieval_result.get("documents", [])
    source_type = retrieval_result.get("source", "Unknown")
    
    if not retrieved_docs:
        return "Sistem tidak memiliki informasi yang cukup untuk menjawab pertanyaan ini."

    context_string = format_retrieved_context(retrieved_docs)
    
    print("[LOG] Menjalankan inferensi GRPO...")
    try:
        raw_response = generation_chain.invoke({
            "context": context_string,
            "question": query
        })
        
        clean_response = clean_model_output(raw_response)
        
    except Exception as e:
        return f"[ERROR] Mesin generasi gagal: {str(e)}"

    citations_string = extract_and_format_citations(retrieved_docs, source_type)
    
    final_output = f"{clean_response}\n{citations_string}"
    print("[SUCCESS] Output final berhasil disusun.")
    
    return final_output

# ==============================================================================
# 3. UJI COBA PIPELINE GENERASI AKHIR (5 TEST CASES)
# ==============================================================================
print("\n" + "="*80)
print("AUDIT GENERASI JAWABAN DAN SITASI (STRICT REASONING MODE)")
print("="*80)

# Mendefinisikan ulang test cases agar sel ini mandiri
test_cases = [
    "Berapa pesangon untuk PHK karena efisiensi menurut PP 35 Tahun 2021?", 
    "Apa syarat sahnya sebuah perjanjian kerja waktu tertentu?",             
    "Siapa Presiden Indonesia yang menjabat pada tahun 2024?",              
    "Berapa harga emas antam 1 gram hari ini?",                             
    "Bagaimana resep dan cara membuat kue bolu panggang yang lembut?"       
]

try:
    # Asumsi variabel 'results' dari Bab 5.3 masih ada di memori
    for i, (query, retrieval_data) in enumerate(zip(test_cases, results), 1):
        print(f"\n--- TEST CASE {i} ---")
        final_answer = execute_rag_generation_pipeline(query, retrieval_data, rag_chain)
        print("\n[OUTPUT FINAL]\n" + final_answer + "\n" + "-"*80)

except NameError as e:
    print(f"\n[FATAL ERROR] Variabel tidak ditemukan: {str(e)}")
    print("[HELP] Pastikan Anda telah menjalankan sel pengujian di Bab 5.3 terlebih dahulu.")


AUDIT GENERASI JAWABAN DAN SITASI (STRICT REASONING MODE)

--- TEST CASE 1 ---

[SYSTEM] Memulai fase generasi untuk: 'Berapa pesangon untuk PHK karena efisiensi menurut...'
[LOG] Menjalankan inferensi GRPO...
[SUCCESS] Output final berhasil disusun.

[OUTPUT FINAL]
<think>Analisis</think>
Jawaban: Pesangon untuk PHK karena efisiensi menurut PP 35 Tahun 2021 adalah setara dengan 0,5 x gaji rata-rata bulanan dari semua pekerja yang diberhentikan. Gaji rata-rata ini dihitung berdasarkan jumlah gaji total dibagi dengan jumlah hari kerja yang dilakukan oleh semua pekerja yang diberhentikan dikalikan dengan jumlah hari kerjanya. Jika ada beberapa jenis gaji, maka digunakan nilai tertinggi dari semua jenis gaji tersebut. Selain itu, jika ada lebih dari satu pekerja yang diberhentikan pada posisi yang sama, maka akan menggunakan gaji rata-rata dari semua pekerja yang diberhentikan pada posisi yang sama tersebut. 

Berikut contoh kasus:

Misalkan sebuah perusahaan memiliki 10 orang karyawan y

## **BAB 7: ANTARMUKA PENGGUNA INTERAKTIF (Interactive User Interface)**

### **7.1. Pembungkusan Pipeline RAG (RAG Pipeline Encapsulation**

Tahap ini mengkonsolidasikan seluruh arsitektur yang telah dibangun dari Bab 2 hingga Bab 6 ke dalam satu fungsi orkestrasi tunggal. Fungsi ini bertindak sebagai *backend API* yang akan dipanggil oleh antarmuka pengguna, menyembunyikan kompleksitas *retrieval*, *reranking*, dan *generation* dari pengguna akhir.

In [18]:
def ask_legal_bot(user_query: str) -> str:
    """
    Fungsi orkestrasi utama (Wrapper) untuk sistem Advanced RAG.
    Menerima input teks dari pengguna dan mengembalikan jawaban akhir yang telah 
    melalui proses HyDE, Hybrid Search, Cross-Encoder Reranking, dan LLM Generation.

    Args:
        user_query (str): Pertanyaan mentah dari pengguna.

    Returns:
        str: Jawaban final dari model lengkap dengan sitasi sumber.
    """
    # Validasi input kosong
    if not user_query or user_query.strip() == "":
        return "Pertanyaan tidak boleh kosong. Silakan masukkan pertanyaan hukum Anda."

    try:
        # 1. Fase Retrieval & Reranking (Bab 5)
        # Menggunakan threshold 0.4 untuk membedakan domain hukum dan non-hukum
        retrieval_data = execute_advanced_retrieval_with_fallback(
            query=user_query,
            hyde_chain=hyde_generator_chain,
            reranker_engine=advanced_reranker_engine,
            threshold=0.4 
        )

        # 2. Fase Generasi & Sitasi (Bab 6)
        final_response = execute_rag_generation_pipeline(
            query=user_query,
            retrieval_result=retrieval_data,
            generation_chain=rag_chain
        )

        return final_response

    except Exception as pipeline_error:
        # Penanganan error tingkat atas untuk mencegah crash pada antarmuka
        error_msg = f"Terjadi kegagalan sistem internal: {str(pipeline_error)}"
        print(f"[FATAL ERROR] {error_msg}")
        return error_msg

### **7.2. Implementasi Antarmuka (Interface Implementation)**

Untuk memenuhi kriteria *Advanced*, kita mengimplementasikan antarmuka menggunakan **Gradio**. Gradio memberikan pengalaman pengguna (UX) berbasis web yang jauh lebih profesional dan interaktif dibandingkan *Python Loop* biasa di dalam terminal.

In [19]:
import re
import gradio as gr

def format_output_for_ui(raw_response: str) -> str:
    """
    Memformat output mentah dari LLM menjadi Markdown yang estetis untuk UI.
    Memisahkan blok penalaran (<think>) dengan jawaban akhir menggunakan 
    elemen visual (seperti blockquote atau italic) agar mudah dibaca.
    """
    # Ekstraksi blok penalaran menggunakan regex
    think_match = re.search(r'<think>(.*?)</think>', raw_response, re.DOTALL)
    
    if think_match:
        reasoning_text = think_match.group(1).strip()
        # Menghapus tag think dari teks utama untuk mendapatkan jawaban akhir
        main_answer = raw_response.replace(f"<think>{reasoning_text}</think>", "").strip()
        
        # Format Markdown: Penalaran diubah menjadi kutipan (blockquote)
        formatted_text = f"### 🧠 Proses Analisis Sistem:\n> *{reasoning_text}*\n\n---\n\n### ⚖️ Jawaban Akhir:\n{main_answer}"
        return formatted_text
    else:
        # Fallback jika tag think tidak ditemukan atau terpotong
        return f"### ⚖️ Jawaban Akhir:\n{raw_response}"

def gradio_interface_handler(query: str) -> str:
    """
    Fungsi perantara (handler) antara Gradio UI dan backend RAG.
    Menerima input dari textbox, memprosesnya melalui pipeline, dan mengembalikan Markdown.
    """
    raw_answer = ask_legal_bot(query)
    formatted_answer = format_output_for_ui(raw_answer)
    return formatted_answer

# Konfigurasi Antarmuka Gradio
print("[SYSTEM] Membangun antarmuka pengguna interaktif...")

legal_bot_ui = gr.Interface(
    fn=gradio_interface_handler,
    inputs=gr.Textbox(
        lines=3, 
        placeholder="Contoh: Apa syarat sahnya perjanjian kerja waktu tertentu menurut undang-undang?",
        label="Masukkan Pertanyaan Hukum Anda"
    ),
    outputs=gr.Markdown(label="Analisis dan Jawaban Asisten"),
    title="🏛️ Asisten AI Legal Indonesia (Advanced RAG)",
    description=(
        "Sistem ini menggunakan arsitektur **Retrieval-Augmented Generation (RAG)** tingkat lanjut. "
        "Dilengkapi dengan *Hybrid Search* (BM25 + Semantic), *Cross-Encoder Reranking*, dan "
        "mekanisme *Fallback* ke internet untuk pertanyaan di luar domain hukum."
    ),
    theme=gr.themes.Soft(),
    flagging_mode="never", # PERBAIKAN: Menggunakan flagging_mode untuk Gradio v4+
    examples=[
        ["Berapa pesangon untuk PHK karena efisiensi menurut PP 35 Tahun 2021?"],
        ["Apa perbedaan mendasar antara PKWT dan PKWTT?"],
        ["Siapa Presiden Indonesia saat ini?"] # Contoh untuk memicu fallback internet
    ]
)

# Eksekusi peluncuran server lokal Gradio
print("[SUCCESS] Antarmuka siap. Meluncurkan server lokal...")
# share=True akan menghasilkan link publik yang bisa diakses dari luar Kaggle
legal_bot_ui.launch(share=True, debug=False)

[SYSTEM] Membangun antarmuka pengguna interaktif...
[SUCCESS] Antarmuka siap. Meluncurkan server lokal...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://f6fc4bdfb3ea6673b2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
